In [ ]:
!pip install -q groq gradio

import os, json
from getpass import getpass
from groq import Groq
import gradio as gr

# ---------- API key ----------
# Colab: click the 🔑 Secrets icon, add a secret named GROQ_API_KEY, enable notebook access.
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY") or getpass("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")

client = Groq(api_key=GROQ_API_KEY)
MODEL = "openai/gpt-oss-120b"  # alt: "llama-3.1-8b-instant" (faster/cheaper)

SYSTEM_PROMPT = """You are an expert nutritionist agent that judges whether fruits are healthy or unhealthy.
The user gives one or more fruits (comma separated), optionally with health conditions and allergies.
Reply with ONLY a JSON object:
{
  "fruits": [
    {
      "name": "clean fruit name",
      "is_fruit": true or false,
      "verdict": "HEALTHY" | "MODERATE" | "UNHEALTHY",
      "score": integer 0-100,
      "reason": "1-2 sentence explanation of the verdict",
      "serving": "typical serving used (e.g. 1 medium, 100 g)",
      "calories_kcal": number,
      "sugar_g": number,
      "fiber_g": number,
      "vitamin_c_mg": number,
      "potassium_mg": number,
      "benefits": ["1-3 short health benefits"],
      "warnings": ["allergy / condition / safety warnings, empty list if none"],
      "tip": "one practical tip"
    }
  ],
  "summary": "one-sentence overall takeaway"
}
Rules:
- Judge by fiber, sugar, glycemic load, vitamins, calorie density and how the fruit is processed.
- Whole fresh fruits are generally HEALTHY in sensible portions. Dried fruit, syrup-packed fruit,
  candied fruit, and packaged juice are usually MODERATE or UNHEALTHY because of concentrated sugar.
- If an item is not a fruit, set is_fruit=false, verdict="UNHEALTHY", score=0, numbers 0, and say it is not a fruit.
- Account for the user's health conditions and allergies in the verdict and warnings.
- Values are approximate estimates for the stated serving."""


def ask_agent(fruits, conditions, allergies):
    user_msg = (
        f"Fruits: {fruits}\n"
        f"Health conditions: {conditions or 'none'}\nAllergies: {allergies or 'none'}"
    )
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": SYSTEM_PROMPT},
                  {"role": "user", "content": user_msg}],
        temperature=0.2,
        response_format={"type": "json_object"},
    )
    return json.loads(resp.choices[0].message.content)


ICONS = {"HEALTHY": "✅", "MODERATE": "⚠️", "UNHEALTHY": "❌"}


def bullets(items):
    return "\n".join(f"- {i}" for i in items) if items else "- None"


def analyze(fruits, conditions, allergies):
    if not fruits or not fruits.strip():
        return "### Please enter at least one fruit.", "", ""
    try:
        d = ask_agent(fruits.strip(), conditions, allergies)
    except Exception as e:
        return f"### Error\n`{e}`", "", ""

    items = d.get("fruits", [])
    if not items:
        return "### No fruits recognised. Try again.", "", ""

    # Summary header
    head = f"## 🍎 Results\n{d.get('summary', '')}"

    # Comparison table
    rows = ["| Fruit | Verdict | Score | Calories | Sugar | Fiber | Vit C |",
            "|---|---|---|---|---|---|---|"]
    for f in items:
        v = f.get("verdict", "MODERATE")
        rows.append(
            f"| **{f.get('name', '?')}** ({f.get('serving', '')}) "
            f"| {ICONS.get(v, 'ℹ️')} {v} | {f.get('score', 0)}/100 "
            f"| {f.get('calories_kcal', 0)} kcal | {f.get('sugar_g', 0)} g "
            f"| {f.get('fiber_g', 0)} g | {f.get('vitamin_c_mg', 0)} mg |"
        )
    table = "\n".join(rows)

    # Per-fruit details
    details = []
    for f in items:
        v = f.get("verdict", "MODERATE")
        details.append(
            f"### {ICONS.get(v, 'ℹ️')} {f.get('name', '?')} — {v}\n"
            f"{f.get('reason', '')}\n\n"
            f"**Benefits**\n{bullets(f.get('benefits'))}\n\n"
            f"**Warnings**\n{bullets(f.get('warnings'))}\n\n"
            f"💡 *{f.get('tip', '')}*\n"
        )
    extra = "\n---\n".join(details) + (
        "\n\n<sub>Estimates only — not medical advice. "
        "Consult a doctor or dietitian for health conditions.</sub>"
    )
    return head, table, extra


with gr.Blocks(theme=gr.themes.Soft(primary_hue="green"), title="Fruit Health AI Agent") as demo:
    gr.Markdown("# 🍓 Fruit Health AI Agent\n"
                "Enter one or more fruits to find out if they're healthy or unhealthy, with nutrition facts.")
    with gr.Row():
        with gr.Column(scale=1):
            fruits = gr.Textbox(label="Fruits (comma separated)",
                                placeholder="e.g. apple, mango, raisins, guava")
            conditions = gr.Textbox(label="Health conditions (optional)",
                                    placeholder="e.g. diabetes, high BP")
            allergies = gr.Textbox(label="Allergies (optional)",
                                   placeholder="e.g. latex-fruit syndrome, kiwi")
            btn = gr.Button("Analyze 🔍", variant="primary")
            gr.Examples(
                examples=[["Apple, Banana, Orange", "", ""],
                          ["Mango, Jackfruit, Grapes", "diabetes", ""],
                          ["Raisins, Dried dates, Packaged fruit juice", "", ""],
                          ["Guava, Papaya, Tomato ketchup", "", ""]],
                inputs=[fruits, conditions, allergies],
            )
        with gr.Column(scale=2):
            head_out = gr.Markdown()
            table_out = gr.Markdown()
            extra_out = gr.Markdown()

    btn.click(analyze, [fruits, conditions, allergies], [head_out, table_out, extra_out])
    fruits.submit(analyze, [fruits, conditions, allergies], [head_out, table_out, extra_out])

demo.launch(debug=True)  # add share=True only if you want a public link

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··········


/tmp/ipykernel_415/373329809.py:120: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="green"), title="Fruit Health AI Agent") as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://633a18fa3c8d2083c1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
